# Session 12: Designing Better Classes

Data Science with Python, Level 1, Stage 1: Python Foundations

## Learning Objectives

By the end of this session you will be able to:

1. Print objects clearly with `__str__` and `__repr__`.
2. Compare and sort objects with `__eq__` and `__lt__`.
3. Share values between objects with class attributes.
4. Protect data with properties and validation.
5. Create objects from text with a class method.
6. Design a class from a short description.

## Where We Are

In Session 11 you built a `Book` class. It works, but it is not pleasant to use yet. Printing a book shows a strange memory address, two identical books are not considered equal, sorting a list of books fails, and nothing stops someone from setting a negative price.

Python's own types behave well because they define a few **special methods**, whose names start and end with two underscores. Today you will add them to your own classes, and learn the habits that make a class safe and easy to use.

In [ ]:
class Book:
    def __init__(self, title, price):
        self.title = title
        self.price = price

print(Book("Python Basics", 45.0))

## 1. Printing Objects

When you call `print(obj)` or `str(obj)`, Python looks for a method called `__str__` and prints what it returns. `__repr__` is used when an object is shown inside a list or as the last line of a cell; it should look like the code that would create the object. If you define only one, define `__repr__`.

In [ ]:
class Book:
    def __init__(self, title, price, stock=0):
        self.title = title
        self.price = price
        self.stock = stock

    def __str__(self):
        return f"{self.title}, {self.price:.2f} ({self.stock} in stock)"

    def __repr__(self):
        return f"Book({self.title!r}, {self.price}, {self.stock})"

b = Book("Python Basics", 45.0, 12)
print(b)
print([b, Book("SQL in a Week", 29.9)])

## 2. Comparing and Sorting Objects

By default, two objects are equal only if they are the very same object in memory. Define `__eq__` to say when two objects should count as equal, and `__lt__` (less than) to say how to order them. With `__lt__`, Python can sort a list of your objects and find the `min` and `max`.

In [ ]:
class Book:
    def __init__(self, title, price):
        self.title = title
        self.price = price

    def __repr__(self):
        return f"Book({self.title!r}, {self.price})"

    def __eq__(self, other):
        return self.title == other.title

    def __lt__(self, other):
        return self.price < other.price

shelf = [Book("SQL", 29.9),
         Book("Python", 45.0),
         Book("Stats", 38.0)]
print(Book("SQL", 29.9) == Book("SQL", 19.9))
print([b.title for b in sorted(shelf)])
print(max(shelf))

| Method | Called by | Use it to |
| --- | --- | --- |
| `__init__` | `Book(...)` | Set up a new object |
| `__str__` | `print(b)`, `str(b)` | Describe the object for people |
| `__repr__` | Lists, the last line of a cell | Show the object for programmers |
| `__eq__` | `a == b` | Decide when two objects are equal |
| `__lt__` | `a < b`, `sorted`, `min`, `max` | Decide the order of objects |
| `__len__` | `len(obj)` | Report the size of a collection |

## 3. Class Attributes

Attributes set on `self` belong to one object. Attributes written directly in the class body belong to the **class** and are shared by every object. Use them for constants that apply to all objects, such as a tax rate, or to count how many objects have been created.

In [ ]:
class Book:
    TAX_RATE = 0.15
    count = 0

    def __init__(self, title, price):
        self.title = title
        self.price = price
        Book.count += 1

    def price_with_tax(self):
        return round(self.price * (1 + Book.TAX_RATE), 2)

b1 = Book("Python Basics", 45.0)
b2 = Book("SQL in a Week", 29.9)
print(b1.price_with_tax(), b2.price_with_tax())
print("Books created:", Book.count)

## 4. Protecting Data

A **property** looks like a normal attribute from outside, but runs a method when it is read or changed. That lets the class check every new value. By convention, a name that starts with one underscore, such as `_price`, means "internal: use the property instead". Python does not enforce this, but every Python programmer respects it.

In [ ]:
class Book:
    def __init__(self, title, price):
        self.title = title
        self.price = price

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("price cannot be negative")
        self._price = value

b = Book("Python Basics", 45.0)
b.price = 39.0
print(b.price)

In [ ]:
b.price = -10

## 5. Alternative Constructors

Data often arrives as text, such as a line from a CSV file. A **class method**, marked with `@classmethod`, receives the class itself as `cls` instead of an object. It is used to write extra ways of creating objects, such as `Book.from_csv(line)`. pandas uses this pattern a lot, for example `pd.DataFrame.from_dict(...)`.

In [ ]:
class Book:
    def __init__(self, title, price, stock):
        self.title, self.price, self.stock = title, price, stock

    @classmethod
    def from_csv(cls, line):
        title, price, stock = line.strip().split(",")
        return cls(title, float(price), int(stock))

    def __repr__(self):
        return f"Book({self.title!r}, {self.price}, {self.stock})"

lines = ["Python Basics,45.0,12", "SQL in a Week,29.9,7"]
books = [Book.from_csv(line) for line in lines]
print(books)

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Prints an address | Forgetting `__str__` or `__repr__` | Define `__repr__` in every class you write. |
| RecursionError | Writing `self.price = value` inside the price setter | Store the value in `self._price` instead. |
| TypeError | Sorting objects without `__lt__` | Define `__lt__`, or sort with `key=lambda b: b.price`. |
| Shared by accident | A list as a class attribute changed by one object | Create lists inside `__init__` on `self`. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: A Printable Point

Write a class `Point` with `x` and `y`. Add `__repr__` that returns `Point(3, 4)`, `__eq__` that compares both coordinates, and a method `distance_to(other)`. Test them.

In [ ]:
# Write your code below

### In-Class Exercise 2: Sortable Students

Write `Student` with `name` and `gpa`. Add `__str__` and `__lt__` so that `sorted` orders students from the lowest to the highest GPA. Sort four students and print them from the highest GPA down.

In [ ]:
# Write your code below

### In-Class Exercise 3: A Safe Thermostat

Write `Thermostat` with a `target` temperature property that only accepts values from 10 to 30, raising `ValueError` otherwise. Set it to 22, then try 45 inside `try` and print the error message.

In [ ]:
# Write your code below

### In-Class Exercise 4: Count the Objects

Add a class attribute `created` to a class `Ticket` that counts how many tickets were made, and give each ticket a `number` from that count. Create three tickets and print their numbers and the total.

In [ ]:
# Write your code below

## Summary

- `__str__` describes an object for people; `__repr__` for programmers.
- `__eq__` defines `==`; `__lt__` makes sorting work.
- Class attributes are shared by every object.
- Properties check values every time they change.
- `@classmethod` gives extra ways to create objects.
- Nouns are attributes, verbs are methods, rules are checks.

## What's Next

A bookshop sells printed books, e-books and audiobooks. They share most of their data but differ in a few details. Next session you will build related classes that inherit from one parent instead of copying code.